In [1]:
import numpy as np

mock_star_window = np.array([
    [15,  20,  25,  20,  15],
    [20,  45,  90,  55,  20],
    [25,  90, 255, 140,  30],
    [20,  55, 140, 110,  25],
    [15,  20,  30,  25,  15]
])

* Creates a fake star window grid array with photon flux integer values to mock photon flux values of viable m33 fits images and the simulated star brightness densities of the core

In [2]:
total_intensity = np.sum(mock_star_window)
y_indices, x_indices = np.indices(mock_star_window.shape)

centroid_x = np.sum(x_indices * mock_star_window) / total_intensity
centroid_y = np.sum(y_indices * mock_star_window) / total_intensity

print("mock centroid tracker live")
print(f"total window intensity: {total_intensity}")
print(f"centroid x value: {centroid_x:.4f} pixels")
print(f"centroid y value: {centroid_y:.4f} pixels")

mock centroid tracker live
total window intensity: 1320
centroid x value: 2.1061 pixels
centroid y value: 2.1061 pixels


* Uses a localized simulated 5x5 mock array to extract the stellar centroid positioning based off of stellar pixel photon brightness values
* Extracts total window intensity (integer) and centroid x and y values (float) to map the center of the frame.

The math: Computes the aggregate scalar mass denominator across the coordinate matrix boundary
$$I_{\text{total}} = \sum_{i=1}^{5} \sum_{j=1}^{5} \mathbf{P}_{i,j}$$

Generates spatial coordinate arrays across vertical $i$ and horizontal $j$ axis
$$i, j \in \text{indices}(\mathbf{P})$$

Calculates precise horizontal centroid coordinates down to sub-pixel values
$$X_c = \frac{\sum_{i=1}^{5} \sum_{j=1}^{5} j \cdot \mathbf{P}_{i,j}}{I_{\text{total}}}$$

and calculates the precise vertical centroid coordinates down to sub-pixel values
$$Y_c = \frac{\sum_{i=1}^{5} \sum_{j=1}^{5} i \cdot \mathbf{P}_{i,j}}{I_{\text{total}}}$$

In [5]:
frame_base = np.zeros((5, 5))
frame_base[2, 2] = 255

frame_drifted = np.zeros((5, 5))
frame_drifted[3, 4] = 255
print(f"base frame start location: {np.argwhere(frame_base == 255)[0]}")
print(f"drifted frame star location: {np.argwhere(frame_drifted == 255)[0]}")

base frame start location: [2 2]
drifted frame star location: [3 4]


* Generates two seperate two-dimensional coordinate arrays modeling a base frame and a star-drifted frame, then tracking photon count variants in drifted star to identify tracking lag
  
The math: Initialize a baseline coordinate matrix with a distinct point signal:
$$\mathbf{F}_{\text{base}} \in \mathbb{Z}^{5 \times 5}, \quad \mathbf{F}_{\text{base}}[2, 2] = 255$$

Initialize a second tracking matrix with a displaced pulse signal:
$$\mathbf{F}_{\text{drifted}} \in \mathbb{Z}^{5 \times 5}, \quad \mathbf{F}_{\text{drifted}}[3, 4] = 255$$

And extract the spatial index coordinates of each respective pulse signal

In [10]:
def calculate_centroid(frame):
    total_intensity = np.sum(frame)
    if total_intensity == 0:
        return 0.0, 0.0
    y_indices, x_indices = np.indices(frame.shape)
    centroid_x = np.sum(x_indices * frame) / total_intensity
    centroid_y = np.sum(y_indices * frame) / total_intensity
    return centroid_x, centroid_y

print(f"CENTROID UTILITY DEPLOYED INTO CORE MEMORY")

CENTROID UTILITY DEPLOYED INTO CORE MEMORY


* Captures the complete intensity-weighted grid math with a single reusable execution script to protect against zero-intensity data short circuits

The math: Create a safety gate to block division by zero mathematical errors if intensity equals 0:
$$\text{if } I_{\text{total}} = 0 \implies (\bar{x}, \bar{y}) = (0.0, 0.0)$$

Compute the complete fractional center of mass coordinates along the horizontal and vertical matrix grids:
$$\bar{x} = \frac{\sum j \cdot \mathbf{P}_{i,j}}{I_{\text{total}}} \quad \text{and} \quad \bar{y} = \frac{\sum i \cdot \mathbf{P}_{i,j}}{I_{\text{total}}}$$

In [12]:
base_x, base_y = calculate_centroid(frame_base)
drifted_x, drifted_y = calculate_centroid(frame_drifted)

print("sub pixel tracking target locations:")
print(f"frame 1 center of mass: X: {base_x:.1f}, Y: {base_y:.1f}")
print(f"frame 2 center of mass: X: {drifted_x:.1f}, Y: {drifted_y:.1f}")

sub pixel tracking target locations:
frame 1 center of mass: X: 2.0, Y: 2.0
frame 2 center of mass: X: 4.0, Y: 3.0


* Executes analytical function calls across prior code blocks to compute and output the exact decimal intensity centers of each frame

The math: maps discrete image matrices into unique subpixel coordinate sets using the centroid calculation procedure:
$$f(\mathbf{F}_{\text{base}}) \implies (X_{\text{base}}, Y_{\text{base}})$$

Maps shifted image matrices into unique subpixel coordinate sets using similar centroid calculation:
$$f(\mathbf{F}_{\text{drifted}}) \implies (X_{\text{drifted}}, Y_{\text{drifted}})$$


In [17]:
delta_x = base_x - drifted_x
delta_y = base_y - drifted_y

aligned_frame = np.roll(frame_drifted, shift=(int(delta_y), int(delta_x)), axis = (0, 1))

print("vector correction verified:")
print(f"calculated shift vectors -> Delta X: {delta_x:.1f}, Delta Y: {delta_y:.1f}")
print(f"frames match verification? {np.array_equal(aligned_frame, frame_base)}")

vector correction verified:
calculated shift vectors -> Delta X: -2.0, Delta Y: -1.0
frames match verification? True


* Calculates relative two dimensional spatial translation offsets and processes an element wise geometric matrix roll to achieve frame alignment

The math: Derive the horizontal and vertical coordinate displacement delta integers across tracking fields:
$$\Delta X = X_{\text{base}} - X_{\text{drifted}} \quad \text{and} \quad \Delta Y = Y_{\text{base}} - Y_{\text{drifted}}$$

Deploy fractional displacement vectors to integers to map structural changes across pixel grids:
$$\vec{s} = (\lfloor \Delta Y \rfloor, \lfloor \Delta X \rfloor)$$

Execute 